In [3]:
from google.colab import files
uploaded = files.upload()

Saving away_team_score.csv to away_team_score.csv
Saving event.csv to event.csv
Saving home_team.csv to home_team.csv
Saving home_team_score.csv to home_team_score.csv


**Question5**

In [5]:
import pandas as pd


file_path = "home_team_score.csv"

df = pd.read_csv(file_path)

period_cols = ["period_1", "period_2", "period_3", "period_4", "period_5"]

df["sets_played"] = df[period_cols].notna().sum(axis=1)

df_valid = df[df["sets_played"] > 0]


print("\nAverage number of sets:", round(df_valid["sets_played"].mean(), 2))
print("Mode (most frequent) number of sets:", df_valid["sets_played"].mode()[0])
print("Median number of sets:", df_valid["sets_played"].median())


Average number of sets: 2.26
Mode (most frequent) number of sets: 2
Median number of sets: 2.0


**Question6**

In [10]:
from google.colab import files
uploaded = files.upload()

Saving away_team.csv to away_team.csv


In [14]:
import pandas as pd

home_df = pd.read_csv("home_team.csv")
away_df = pd.read_csv("away_team.csv")
event_df = pd.read_csv("event.csv")


players_df = pd.concat([home_df, away_df], ignore_index=True)
players_unique = players_df.drop_duplicates(subset="player_id").dropna(subset=["country"])
players_unique["total_prize"] = pd.to_numeric(players_unique["total_prize"], errors="coerce")
players_unique["current_rank"] = pd.to_numeric(players_unique["current_rank"], errors="coerce")


home_wins = event_df[event_df["winner_code"] == 1][["match_id"]].merge(
    home_df[["match_id", "player_id"]], on="match_id", how="left"
)
away_wins = event_df[event_df["winner_code"] == 2][["match_id"]].merge(
    away_df[["match_id", "player_id"]], on="match_id", how="left"
)

all_wins = pd.concat([home_wins, away_wins], ignore_index=True)
wins_per_player = all_wins.groupby("player_id").size().reset_index(name="wins")


players_unique = players_unique.merge(wins_per_player, on="player_id", how="left")
players_unique["wins"] = players_unique["wins"].fillna(0)


country_stats = players_unique.groupby("country").agg(
    player_count=("player_id", "count"),
    avg_rank=("current_rank", "mean"),
    total_wins=("wins", "sum"),
    total_prize=("total_prize", "sum"),
).reset_index()


country_stats["rank_score"] = 1 - (
    (country_stats["avg_rank"] - country_stats["avg_rank"].min())
    / (country_stats["avg_rank"].max() - country_stats["avg_rank"].min())
)
country_stats["wins_score"] = (
    country_stats["total_wins"] - country_stats["total_wins"].min()
) / (country_stats["total_wins"].max() - country_stats["total_wins"].min())
country_stats["prize_score"] = (
    country_stats["total_prize"] - country_stats["total_prize"].min()
) / (country_stats["total_prize"].max() - country_stats["total_prize"].min())

country_stats["success_score"] = (
    country_stats["rank_score"] + country_stats["wins_score"] + country_stats["prize_score"]
) / 3


result = country_stats.sort_values("success_score", ascending=False)


print("most successful country in tennis (combined score):")
print(result[["country", "player_count", "avg_rank", "total_wins", "total_prize", "success_score"]].head(1).to_string(index=False))

most successful country in tennis (combined score):
country  player_count  avg_rank  total_wins  total_prize  success_score
    USA           225 681.68018      3922.0  218863566.0       0.834411
